## Example 1: Functionality Test 1

In [ ]:
import joserfc.jwt
import joserfc.jwk
import time

# Example 1: JWT encode() and decode() functions
print("Example 1: JWT encode() and decode() Functions")

# Generate a proper key using JWK (without size parameter)
try:
    secret_key = joserfc.jwk.generate_key("oct")
    print(f"Generated key type: {type(secret_key)}")
except Exception as e:
    print(f"Key generation error: {e}")
    # Try alternative approach - create key from bytes
    import secrets
    key_bytes = secrets.token_bytes(32)  # 32 bytes = 256 bits
    secret_key = joserfc.jwk.guess_key({"kty": "oct", "k": joserfc.jwk.import_key(key_bytes).get("k")})
    print(f"Alternative key creation successful")

# Create JWT payload with standard claims
# NOTE: `jwt.decode()` returns a Token whose claims live on `.claims` — there is no `.payload`.
payload = {
    "sub": "user123",
    "name": "Alice Johnson",
    "email": "alice@example.com", 
    "iat": int(time.time()),
    "exp": int(time.time()) + 3600,  # Expires in 1 hour
    "role": "admin"
}

# JWT header
header = {"alg": "HS256", "typ": "JWT"}

try:
    # Encode JWT using joserfc.jwt.encode with proper key
    token = joserfc.jwt.encode(header, payload, secret_key)
    print(f"Encoded JWT: {token[:50]}...{token[-20:]}")

    # Decode JWT using joserfc.jwt.decode
    decoded_token = joserfc.jwt.decode(token, secret_key)
    print(f"Decoded header: {decoded_token.header}")
    print(f"Decoded payload: {decoded_token.claims}")
    print(f"User: {decoded_token.claims['name']} ({decoded_token.claims['role']})")

    # Access token properties
    print(f"Token subject: {decoded_token.claims['sub']}")
    print(f"Time until expiration: {decoded_token.claims['exp'] - int(time.time())} seconds")

    # Test with invalid signature: decode must reject it with BadSignatureError
    invalid_token = token[:-5] + "XXXXX"  # Corrupt the signature
    try:
        joserfc.jwt.decode(invalid_token, secret_key)
    except joserfc.errors.BadSignatureError as e:
        print(f"Invalid signature caught: {type(e).__name__}")
    else:
        raise AssertionError("a JWT with a corrupted signature decoded without error")

    # Test different algorithm
    header_hs512 = {"alg": "HS512", "typ": "JWT"}
    # joserfc ships a RECOMMENDED-only algorithm registry by default, and HS512 is not on it —
    # `UnsupportedAlgorithmError: Algorithm of 'HS512' is not recommended`. Opting in is explicit,
    # and the same registry must be supplied on the way back out.
    hs512_registry = joserfc.jws.JWSRegistry(algorithms=["HS512"])
    token_hs512 = joserfc.jwt.encode(header_hs512, payload, secret_key, registry=hs512_registry)
    decoded_hs512 = joserfc.jwt.decode(token_hs512, secret_key, registry=hs512_registry)
    print(f"HS512 algorithm: {decoded_hs512.header['alg']}")

except Exception as e:
    print(f"JWT operation error: {type(e).__name__}: {e}")
    raise AssertionError(
        f"joserfc example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed successfully\n")

## Example 2: Functionality Test 2

In [ ]:
import joserfc.jws
import joserfc.jwk
import json

# Example 2: JWS (JSON Web Signature) with serialize_compact and deserialize_compact
print("Example 2: JWS with serialize_compact and deserialize_compact")

# Generate proper signing key for JWS
signing_key = joserfc.jwk.generate_key("oct")
print(f"Generated signing key type: {type(signing_key)}")

# Create message payload
message_payload = {"user": "alice", "action": "login", "timestamp": 1234567890}

# Create protected header with HS256 algorithm
protected_header = {"alg": "HS256", "typ": "JWS"}
payload_bytes = json.dumps(message_payload).encode('utf-8')

# Sign and serialize to compact format (protected, payload, key)
jws_token = joserfc.jws.serialize_compact(protected_header, payload_bytes, signing_key)
print(f"JWS Token: {jws_token[:50]}...")

# Verify and deserialize the JWS token
verified_data = joserfc.jws.deserialize_compact(jws_token, signing_key)
print(f"Verified payload: {json.loads(verified_data.payload)}")
print(f"Protected header alg: {verified_data.protected['alg']}")

# Create another JWS with different payload
admin_payload = {"user": "admin", "role": "superuser", "exp": 9999999999}
admin_payload_bytes = json.dumps(admin_payload).encode('utf-8')
admin_jws = joserfc.jws.serialize_compact(protected_header, admin_payload_bytes, signing_key)
admin_verified = joserfc.jws.deserialize_compact(admin_jws, signing_key)
print(f"Admin payload verified: {json.loads(admin_verified.payload)}")

print("Example 2 completed successfully\n")

## Example 3: Functionality Test 3

In [ ]:
import joserfc.jwe
import joserfc.jwk
import json

# Example 3: JWE (JSON Web Encryption) with encrypt_compact and decrypt_compact
print("Example 3: JWE with encrypt_compact and decrypt_compact")

# Generate proper encryption key for JWE (RSA for better compatibility)
try:
    encryption_key = joserfc.jwk.generate_key("RSA")
    print(f"Generated RSA encryption key type: {type(encryption_key)}")
except:
    # Fallback to oct key if RSA fails
    encryption_key = joserfc.jwk.generate_key("oct")
    print(f"Generated oct encryption key type: {type(encryption_key)}")

# Create sensitive payload to encrypt
sensitive_payload = {"ssn": "123-45-6789", "credit_card": "4111-1111-1111-1111", "pin": "1234"}
payload_bytes = json.dumps(sensitive_payload).encode('utf-8')

# Create protected header for JWE
try:
    # Try RSA-OAEP with A256GCM
    protected_header = {"alg": "RSA-OAEP", "enc": "A256GCM"}
    
    # Encrypt using encrypt_compact (protected, plaintext, public_key)
    jwe_token = joserfc.jwe.encrypt_compact(protected_header, payload_bytes, encryption_key)
    print(f"JWE Token: {jwe_token[:50]}...")
    
    # Decrypt using decrypt_compact
    decrypted_data = joserfc.jwe.decrypt_compact(jwe_token, encryption_key)
    print(f"Decrypted payload: {json.loads(decrypted_data.plaintext)}")
    print(f"Protected header alg: {decrypted_data.protected['alg']}")
    
except Exception as e:
    print(f"RSA encryption failed: {e}")
    # Fallback to direct encryption with oct key
    try:
        protected_header = {"alg": "dir", "enc": "A256GCM"}
        jwe_token = joserfc.jwe.encrypt_compact(protected_header, payload_bytes, encryption_key)
        decrypted_data = joserfc.jwe.decrypt_compact(jwe_token, encryption_key)
        print(f"Direct encryption successful")
        print(f"Decrypted payload: {json.loads(decrypted_data.plaintext)}")
        print(f"Protected header: {decrypted_data.protected}")
    except Exception as e2:
        print(f"All encryption methods failed: {e2}")
        raise

print("Example 3 completed successfully\n")

## Example 4: Functionality Test 4

In [ ]:
import joserfc.jwk

# Example 4: JWK generate_key() and key management functions
print("Example 4: JWK generate_key() and Key Management")

# Generate symmetric key (oct - octet sequence)
try:
    symmetric_key = joserfc.jwk.generate_key("oct")
    print(f"Generated symmetric key type: {type(symmetric_key)}")
    print(f"Key type (kty): {symmetric_key.get('kty')}")
    print(f"Key use: {symmetric_key.get('use', 'Not specified')}")
    print(f"Key algorithm: {symmetric_key.get('alg', 'Not specified')}")
    
    # Export key as dictionary
    sym_dict = symmetric_key.as_dict()
    print(f"Symmetric key dict keys: {list(sym_dict.keys())}")
    
except Exception as e:
    print(f"Symmetric key generation failed: {e}")
    raise

# Generate RSA key pair
try:
    rsa_keypair = joserfc.jwk.generate_key("RSA")
    print(f"\nGenerated RSA key type: {type(rsa_keypair)}")
    
    # Check key components
    rsa_dict = rsa_keypair.as_dict()
    has_private = 'd' in rsa_dict  # 'd' is the RSA private exponent
    print(f"RSA key has private components: {has_private}")
    print(f"RSA key components: {sorted(rsa_dict.keys())}")
    
    # Extract public key components from private key (e and n are public)
    public_components = ['kty', 'n', 'e']
    public_dict = {k: rsa_dict[k] for k in public_components if k in rsa_dict}
    print(f"Public key components: {sorted(public_dict.keys())}")
    print(f"Public key modulus (n) length: {len(rsa_dict.get('n', ''))}")
    
    # Check if key is private
    print(f"Is private key: {rsa_keypair.is_private}")
    
except Exception as e:
    print(f"RSA key generation failed: {e}")
    raise

# Import existing key using import_key
try:
    # Test key dictionary (RFC 7517 example) 
    existing_key_dict = {
        "kty": "oct",
        "k": "AyM1SysPpbyDfgZld3umj1qzKObwVMkoqQ-EstJQLr_T-1qS0gZH75aKtMN3Yj0iPS4hcgUuTwjAzZr1Z9CAow",
        "use": "sig",
        "alg": "HS256"
    }
    
    imported_key = joserfc.jwk.import_key(existing_key_dict)
    print(f"\nImported key type: {type(imported_key)}")
    print(f"Import successful: {imported_key is not None}")
    print(f"Imported key algorithm: {imported_key.get('alg')}")
    
except Exception as e:
    print(f"Key import failed: {e}")
    raise

# Key thumbprint generation
try:
    if 'symmetric_key' in locals():
        thumbprint = joserfc.jwk.thumbprint(symmetric_key)
        print(f"Key thumbprint: {thumbprint}")
        
except Exception as e:
    print(f"Thumbprint generation failed: {e}")
    raise

print("Example 4 completed successfully\n")

## Example 5: Functionality Test 5

In [ ]:
import joserfc.jwt
import joserfc.jwk
import joserfc.errors
import time

# Example 5: JWT Claims validation and advanced features
print("Example 5: JWT Claims Validation and Advanced Features")

# Generate proper key for JWT signing
secret_key = joserfc.jwk.generate_key("oct")
print(f"Generated secret key type: {type(secret_key)}")

# Create JWT with comprehensive claims
current_time = int(time.time())

# Payload with standard and custom claims
payload = {
    "iss": "auth.example.com",           # Issuer
    "sub": "user789",                    # Subject
    "aud": ["api.example.com"],          # Audience 
    "exp": current_time + 1800,          # Expires (30 minutes)
    "nbf": current_time - 60,            # Not before (1 minute ago)
    "iat": current_time,                 # Issued at
    "jti": "unique-jwt-id-456",          # JWT ID
    "role": "user",
    "permissions": ["read", "write"],
    "session_id": "sess_123456"
}

header = {"alg": "HS256", "typ": "JWT"}

# Encode JWT with claims using proper key
jwt_with_claims = joserfc.jwt.encode(header, payload, secret_key)
print(f"JWT with claims: {jwt_with_claims[:30]}...{jwt_with_claims[-20:]}")

# Decode: jwt.decode verifies the SIGNATURE only; the claims are validated by JWTClaimsRegistry below
decoded_jwt = joserfc.jwt.decode(jwt_with_claims, secret_key)
print(f"Decoded token type: {type(decoded_jwt)}")

# Access claims from the token using .claims attribute
claims = decoded_jwt.claims
print(f"Issuer: {claims['iss']}")
print(f"Subject: {claims['sub']}")
print(f"Audience: {claims['aud']}")
print(f"Expires in: {claims['exp'] - current_time} seconds")

# Validate the registered claims: validate() returns None when they hold and raises a ClaimError subclass otherwise
claims_registry = joserfc.jwt.JWTClaimsRegistry(
    iss={"essential": True, "value": "auth.example.com"},
    aud={"essential": True, "value": "api.example.com"},
)
claims_registry.validate(claims)
print("Fresh token claims validated (iss, aud, exp, nbf, iat)")

# Validate custom claims
if claims.get('role') == 'user':
    print("User role validated")

permissions = claims.get('permissions', [])
if 'read' in permissions:
    print("Read permission granted")

# Access header information
print(f"Token algorithm: {decoded_jwt.header['alg']}")
print(f"Token type: {decoded_jwt.header['typ']}")

# Test expired token: its signature is valid, so jwt.decode accepts it; the claims registry must REJECT it
expired_payload = payload.copy()
expired_payload['exp'] = current_time - 3600  # Expired 1 hour ago
expired_jwt = joserfc.jwt.encode(header, expired_payload, secret_key)
expired_token = joserfc.jwt.decode(expired_jwt, secret_key)
try:
    claims_registry.validate(expired_token.claims)
except joserfc.errors.ExpiredTokenError as e:
    print(f"Expired token correctly rejected: {type(e).__name__}")
else:
    raise AssertionError("JWTClaimsRegistry.validate accepted a token that expired an hour ago")

# Check JWT token structure
token_parts = jwt_with_claims.split('.')
print(f"JWT structure: {len(token_parts)} parts (header.payload.signature)")

print("Example 5 completed successfully\n")